# Treasure Hunt Game

**Author: Alysha Pursley**

**Purpose:** This notebook is the executable orchestration layer for the original Python/Jupyter project. It defines the maze data, visualization helpers, neural-network model, deep Q-learning training loop, completion check, and final model test while importing environment and replay behavior from the two helper modules.

The Treasure Hunt Game uses an 8×8 maze in which a pirate agent learns to reach the treasure through deep Q-learning.

## Source Structure

`treasure_maze.py` defines the maze environment and movement/reward rules. `game_experience.py` stores state transitions for experience replay. This notebook defines the model, training loop, completion check, and final game tests.

The original project materials identify the two helper modules as supplied starter code. Exact supplied copies are preserved under `starter-code/` in the packaged project.

A public qmaze tutorial contains closely similar maze and replay structure and is recorded in the repository attribution documentation as an external comparison. The available original project materials do not explicitly cite qmaze, so the repository does not claim a confirmed direct provenance chain.


In [ ]:
# Treasure Hunt Game | Original/treasure_hunt_game.ipynb
# Purpose: Contains the original notebook training workflow, visualizations,
#   and model evaluation.
# Author: Alysha Pursley
# Copyright (c) 2026 Alysha Pursley.
# License: See LICENSE (all rights reserved).
from __future__ import print_function
import datetime
import random

import matplotlib.pyplot as plt
import numpy as np
from keras.layers import Dense, Input, PReLU
from keras.models import Sequential

from treasure_maze import TreasureMaze
from game_experience import GameExperience

%matplotlib inline


The 8×8 matrix below is the project maze used by the environment and training workflow.


In [ ]:
# 1.0 cells are traversable and 0.0 cells are walls. Keeping the maze as a
# numeric matrix lets the same structure drive both environment logic and model input.
maze = np.array([
    [ 1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  0.,  1.,  1.,  1.,  0.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  0.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  0.,  0.,  0.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.]
])

The visualization helper renders maze walls, open cells, the pirate position, and the treasure during development and model checks.


In [ ]:
def show(qmaze):
    plt.grid('on')
    nrows, ncols = qmaze.maze.shape
    ax = plt.gca()
    ax.set_xticks(np.arange(0.5, nrows, 1))
    ax.set_yticks(np.arange(0.5, ncols, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    # Draw from a copy so visualization markers never alter environment state.
    canvas = np.copy(qmaze.maze)
    for row,col in qmaze.visited:
        canvas[row,col] = 0.6
    pirate_row, pirate_col, _ = qmaze.state
    canvas[pirate_row, pirate_col] = 0.3   # Pirate marker.
    canvas[nrows-1, ncols-1] = 0.9        # Treasure marker.
    img = plt.imshow(canvas, interpolation='none', cmap='gray')
    return img

The pirate agent can move in four directions: left, right, up, and down.

I use epsilon to balance exploitation and exploration. With an epsilon of 0.1, the agent usually follows the action with the highest predicted value but occasionally chooses a random valid action so it can discover alternatives.


In [ ]:
LEFT = 0
UP = 1
RIGHT = 2
DOWN = 3


# Probability of choosing a random valid action during training.
epsilon = 0.1

# Keep the numeric action IDs aligned with the environment helper.
actions_dict = {
    LEFT: 'left',
    UP: 'up',
    RIGHT: 'right',
    DOWN: 'down',
}

num_actions = len(actions_dict)


This environment check creates the maze, performs one downward action, reports its reward, and visualizes the resulting state.


In [ ]:
qmaze = TreasureMaze(maze)
canvas, reward, game_over = qmaze.act(DOWN)
print("reward=", reward)
show(qmaze)

This function simulates a full game based on the trained model. The other parameters include the TreasureMaze object and the starting position of the pirate.

In [ ]:
def play_game(model, qmaze, pirate_cell):
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    # Evaluation is greedy: exploration belongs in training, not the completion test.
    while True:
        prev_envstate = envstate
        # Choose the highest-valued action predicted by the trained model.
        q = model.predict(prev_envstate, verbose=0)
        action = np.argmax(q[0])

        # Advance the environment until this start position wins or loses.
        envstate, reward, game_status = qmaze.act(action)
        if game_status == 'win':
            return True
        elif game_status == 'lose':
            return False

This helper checks whether the current model can reach the treasure from every valid starting cell. The maze is designed with at least one valid route to the target.


In [ ]:
def completion_check(model, qmaze):
    # A model is complete only if it can finish from every usable start cell.
    for cell in qmaze.free_cells:
        # A stranded cell cannot be a meaningful evaluation starting point.
        if not qmaze.valid_actions(cell):
            return False
        if not play_game(model, qmaze, cell):
            return False
    return True

This block builds the neural-network model used for Q-value prediction. The architecture, activation functions, optimizer, and loss function define how the model learns action values from maze states.


In [ ]:
def build_model(maze):
    # The flattened maze is both the input width and hidden-layer width.
    model = Sequential()
    model.add(Input(shape=(maze.size,)))
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(maze.size))
    model.add(PReLU())
    # One output Q-value is produced for each directional action.
    model.add(Dense(num_actions))
    model.compile(optimizer='adam', loss='mse')
    return model

## Deep Q-Learning Training Loop

The training loop resets the maze from valid starting cells, chooses actions through epsilon-greedy exploration and exploitation, stores transitions in replay memory, trains the model from sampled experiences, tracks recent wins, and verifies the learned policy with the completion check.


In [ ]:
def qtrain(model, maze, **opt):

    global epsilon

    # Accept the notebook's existing "epochs" call while retaining "n_epoch"
    # for compatibility with the original training helper.
    n_epoch = opt.get('n_epoch', opt.get('epochs', 15000))
    max_memory = opt.get('max_memory', 1000)
    data_size = opt.get('data_size', 50)
    for name, value in (
        ('n_epoch', n_epoch),
        ('max_memory', max_memory),
        ('data_size', data_size),
    ):
        if isinstance(value, bool) or not isinstance(value, int) or value < 1:
            raise ValueError(f'{name} must be a positive integer.')

    start_time = datetime.datetime.now()

    qmaze = TreasureMaze(maze)
    experience = GameExperience(model, max_memory=max_memory)

    # Use a rolling half-maze window so recent reliability matters more than
    # early exploratory failures when deciding whether training can stop.
    win_history = []
    hsize = qmaze.maze.size // 2
    win_rate = 0.0

    for epoch in range(n_epoch):
        # Randomize the start cell so the model cannot memorize one route only.
        agent_cell = random.choice(qmaze.free_cells)
        qmaze.reset(agent_cell)
        envstate = qmaze.observe()

        loss = 0
        n_episodes = 0

        while True:
            previous_envstate = envstate

            # Choose between exploration and exploitation.
            if np.random.rand() < epsilon:
                action = random.choice(qmaze.valid_actions())
            else:
                action = np.argmax(model.predict(previous_envstate, verbose=0))

            envstate, reward, game_status = qmaze.act(action)
            game_over = game_status in ('win', 'lose')

            # Replay memory expects a terminal boolean, not the status string.
            experience.remember(
                [previous_envstate, action, reward, envstate, game_over]
            )
            n_episodes += 1

            if game_over:
                win_history.append(1 if game_status == 'win' else 0)
                break

            # Train from replayed transitions instead of only the latest move;
            # this reduces correlation between consecutive maze states.
            inputs, targets = experience.get_data(data_size)
            history = model.fit(
                inputs,
                targets,
                epochs=1,
                batch_size=16,
                verbose=0,
            )
            loss += history.history['loss'][0]

        win_rate = sum(win_history[-hsize:]) / hsize

        dt = datetime.datetime.now() - start_time
        t = format_time(dt.total_seconds())
        template = (
            "Epoch: {:03d}/{:d} | Loss: {:.4f} | Episodes: {:d} | "
            "Win count: {:d} | Win rate: {:.3f} | time: {}"
        )
        print(
            template.format(
                epoch,
                n_epoch - 1,
                loss,
                n_episodes,
                sum(win_history),
                win_rate,
                t,
            )
        )

        if win_rate > 0.9:
            # Once wins are consistent, reduce exploration but keep a small
            # chance of trying alternatives until completion is verified.
            epsilon = 0.05

        # Stop only after a perfect recent window and a greedy completion
        # check from every usable start cell.
        if (
            sum(win_history[-hsize:]) == hsize
            and completion_check(model, qmaze)
        ):
            print("Reached 100%% win rate at epoch: %d" % (epoch,))
            break

    dt = datetime.datetime.now() - start_time
    seconds = dt.total_seconds()
    t = format_time(seconds)

    print(
        "n_epoch: %d, max_mem: %d, data: %d, time: %s"
        % (epoch, max_memory, data_size, t)
    )
    return seconds


def format_time(seconds):
    """Return a compact human-readable training duration."""
    if seconds < 400:
        return "%.1f seconds" % float(seconds)
    if seconds < 4000:
        return "%.2f minutes" % (seconds / 60.0)
    return "%.2f hours" % (seconds / 3600.0)


## Testing the Model

The remaining cells create the maze environment, train the model, run the completion check, and test one game from the top-left starting cell.


In [ ]:
qmaze = TreasureMaze(maze)
show(qmaze)

The next block builds the model and starts deep Q-learning training. Training time depends on the machine-learning environment and the number of epochs required before the completion condition is met.


In [ ]:
model = build_model(maze)
qtrain(model, maze, epochs=1000, max_memory=8*maze.size, data_size=32)

This cell runs the completion check across valid starting cells. Execution time depends on the machine-learning runtime and trained model.


In [ ]:
completion_check(model, qmaze)
show(qmaze)

This cell tests one game from the top-left corner. The trained agent should navigate from `(0, 0)` to the treasure in the bottom-right cell.


In [ ]:
pirate_start = (0, 0)
play_game(model, qmaze, pirate_start)
show(qmaze)

## Save the Training Record
This notebook can be saved with its outputs as a record of the training workflow, model behavior, and verification steps.
